In [0]:
# Check that the Databricks environment is working

print("Databricks is working!")

# Show the Spark version
print("Spark version:", spark.version)

In [0]:
%pip install xgboost

In [0]:
df = spark.table("electricity_load")

display(df)

In [0]:
from pyspark.sql import functions as F

# read the raw bronze table 
df = spark.table("electricity_load")

df = (
    df.withColumn("validFrom", F.to_timestamp("validFrom"))
    .withColumn("validto", F.to_timestamp("validto"))
)


# ordering the data by time 

df = df.orderBy("validFrom")

display(df)


In [0]:
from pyspark.sql.window import Window

# window ordered chronologically

time_window = Window.orderBy("validFrom")

# create calendar features

df_features = (
    df.withColumn("hour", F.hour("validFrom"))
    .withColumn("day_of_week", F.dayofweek("validFrom"))
    .withColumn("day_of_month", F.dayofmonth("validFrom"))
    .withColumn("month", F.month("validFrom"))
    .withColumn("year",F.year("validFrom"))
    .withColumn("is_weekend", F.when(F.dayofweek("validFrom").isin([1,7]),1).otherwise(0))
)

# create lag features 
# the paranthesis here tells us that the expression will continue on the next line 
df_features = (
    df_features.withColumn("load_lag_1h", F.lag("volume",1).over(time_window))
    .withColumn("load_lag_24h", F.lag("volume",24).over(time_window))
    .withColumn("load_lag_7d",F.lag("volume",24*7).over(time_window))
)



df_features = (
    df_features.withColumn("trend_feautre",F.col("volume")-F.lag("volume",1).over(time_window))
)

# create rolling average 

rolling_24 = (
    time_window.rowsBetween(-24,-1)
)

rolling_7d = (time_window.rowsBetween(-(24*7),-1))

df_features = (
    df_features.withColumn("load_rolling_avg_24", F.avg("volume").over(rolling_24))
    .withColumn("load_rolling_7d", F.avg("volume").over(rolling_7d))
)

display(df_features)







In [0]:
# Target = electricity load one hour in the future 

df_features = df_features.withColumn("target", F.lead("volume",1).over(time_window))

# Remove rows where the required lag/target values do not exisit 

df_features = df_features.dropna()

display(df_features)


In [0]:
# saving the table as a databricks table 

df_features.write.mode("overwrite").saveAsTable("electricity_load_features")

print("Feature Table Created successfully")

In [0]:
# Read the table back from Databricks
features = spark.table("electricity_load_features")

# Show its structure
features.printSchema()

# Display some rows
display(features.limit(10))

In [0]:
from pyspark.sql import functions as F

# Read the feature table 
df = spark.table("electricity_load_features")

# sort the table chronologically
df.orderBy("validFrom")

# then let us split into train and test
total_rows = df.count()

train_end  = int(total_rows * 0.70)
validation_end  = int(total_rows * 0.85)

# a row number column so we can split the data chronologically
df_numbered = df.withColumn("row_number",F.row_number().over(time_window))


# creating train, test and other types of datasets

# Create the three datasets
train = df_numbered.filter(F.col("row_number") <= train_end)
validation = df_numbered.filter(F.col("row_number") > validation_end )

validation = df_numbered.filter(
    (F.col("row_number") > train_end) &
    (F.col("row_number") <= validation_end)
)

test = df_numbered.filter(
    F.col("row_number") > validation_end
)





# Remove the temporary row number
train = train.drop("row_number")
validation = validation.drop("row_number")
test = test.drop("row_number")

print("Training rows:", train.count())
print("Validation rows:", validation.count())
print("Test rows:", test.count())

In [0]:
print("TRAIN:")
print(train.agg(
    F.min("validfrom").alias("start"),
    F.max("validfrom").alias("end")
).collect()[0])

print("\nVALIDATION:")
print(validation.agg(
    F.min("validfrom").alias("start"),
    F.max("validfrom").alias("end")
).collect()[0])

print("\nTEST:")
print(test.agg(
    F.min("validfrom").alias("start"),
    F.max("validfrom").alias("end")
).collect()[0])

In [0]:
from pyspark.sql import functions as F

# The current load is our prediction for the next hour
validation_baseline = validation.withColumn(
    "baseline_prediction",
    F.col("volume")
)

test_baseline = test.withColumn(
    "baseline_prediction",
    F.col("volume")
)

In [0]:
# target is load one hour into the future

# volume at t --> prediction
# target at t --> actual load at t+1

In [0]:
from pyspark.sql import functions as F

# The current load is our prediction for the next hour
validation_baseline = validation.withColumn(
    "baseline_prediction",
    F.col("volume")
)

test_baseline = test.withColumn(
    "baseline_prediction",
    F.col("volume")
)

In [0]:
validation_mae = validation_baseline.select(F.avg(F.abs(F.col("target") - F.col("baseline_prediction"))).alias("MAE")).collect()[0]['MAE']

# calculating baseline MAE on test set
test_mae = test_baseline.select(F.avg(F.abs(F.col("target") - F.col("baseline_prediction"))).alias("MAE")).collect()[0]['MAE']

print("Baseline validation MAE:", validation_mae)
print("Baseline test MAE:", test_mae)

In [0]:
feature_columns = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "load_lag_1h",
    "load_lag_24h",
    "load_lag_7d",
    "load_rolling_avg_24",
    "load_rolling_7d"
]

target_column = "target"

train_model = train.select(feature_columns + [target_column])
validation_model = validation.select(feature_columns + [target_column])
test_model = test.select(feature_columns + [target_column])

display(train_model.limit(5))

In [0]:
# Convert Spark DataFrames to pandas
train_pd = train_model.toPandas()
validation_pd = validation_model.toPandas()
test_pd = test_model.toPandas()

print("Training shape:", train_pd.shape)
print("Validation shape:", validation_pd.shape)
print("Test shape:", test_pd.shape)

In [0]:
from xgboost import XGBRegressor



# Separate features (X) and target (y)
X_train = train_pd[feature_columns]
y_train = train_pd[target_column]

X_validation = validation_pd[feature_columns]
y_validation = validation_pd[target_column]

# Create the XGBoost regression model
model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    objective="reg:squarederror",
    random_state=42
)

# Train the model
model.fit(
    X_train,
    y_train,
    eval_set=[(X_validation, y_validation)],
    verbose=False
)


print("XGBoost model trained successfully.")

In [0]:
# Generate predictions for the validation period
validation_pd["prediction"] = model.predict(X_validation)

# Look at actual vs predicted values
print(
    validation_pd[
        [target_column, "prediction"]
    ].head(10)
)

In [0]:
from sklearn.metrics import mean_absolute_error
# baseline MAE vs xgboost MAE
# Calculate MAE for the XGBoost model
xgb_mae = mean_absolute_error(
    validation_pd["target"],
    validation_pd["prediction"]
)

print("XGBoost validation MAE:", xgb_mae)

In [0]:
print("Baseline validation MAE:", validation_mae)
print("XGBoost validation MAE:", xgb_mae)

improvement = (
    (validation_mae - xgb_mae)
    / validation_mae
) * 100

print("Improvement:", improvement, "%")

In [0]:
# Use the electricity load from 24 hours earlier
# as the prediction for the next hour
validation_day_baseline = validation.withColumn(
    "baseline_prediction",
    F.col("load_lag_24h")
)

# Calculate MAE
day_baseline_mae = validation_day_baseline.select(
    F.avg(
        F.abs(
            F.col("target") - F.col("baseline_prediction")
        )
    ).alias("MAE")
).collect()[0]["MAE"]

print("Previous-day baseline MAE:", day_baseline_mae)

In [0]:
import pandas as pd
# Get the importance of each feature from the trained XGBoost model
feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": model.feature_importances_
})

# Sort from most important to least important
feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

print(feature_importance)